# Assamese Positional-Embedding Ablation — Kaggle Run

**This notebook is compiled for remote execution.**

Before running: select **GPU → Tesla T4**, then use **Save Version → Save & Run All**. After the versioned run has started, you can close the browser/laptop; do not manually stop the Kaggle session.

The notebook:
1. prepares the Assamese tokenizer/data paths,
2. builds the decoder-only model **without positional embeddings**,
3. trains for the Phase-2 target (~500M tokens / 30,518 optimizer steps),
4. automatically resumes from the latest valid checkpoint if a previous checkpoint exists,
5. evaluates Validation CE/PPL, BPB, standard generation, quantitative generation metrics, and attention,
6. packages the final outputs and latest checkpoints into one directory and ZIP.


In [1]:
from pathlib import Path
import shutil
import json
import os

# ------------------------------------------------------------------
# Robustly locate the Assamese dataset under common Kaggle mount forms.
# The current project uses /kaggle/input/datasets/vidurshah04/assam-data.
# ------------------------------------------------------------------
CANDIDATE_DATA_DIRS = [
    Path("/kaggle/input/datasets/vidurshah04/assam-data"),
]

DATA_DIR = None
for candidate in CANDIDATE_DATA_DIRS:
    if candidate.exists() and list(candidate.glob("train_*.txt")) and list(candidate.glob("val_*.txt")):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate Assamese train/val text files. "
        "Expected /kaggle/input/datasets/vidurshah04/assam-data."
    )

TOKENIZER_SOURCE = DATA_DIR / "assamese_10k.model"
TOKENIZER_PATH = Path("/kaggle/working/assamese_10k.model")

if not TOKENIZER_SOURCE.exists():
    raise FileNotFoundError(
        f"Assamese tokenizer not found at {TOKENIZER_SOURCE}"
    )

shutil.copy2(TOKENIZER_SOURCE, TOKENIZER_PATH)

BASE_DIR = Path("/kaggle/working/assamese_10k")
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
LOG_DIR = BASE_DIR / "logs"
EVAL_DIR = BASE_DIR / "evaluation"

for d in [CHECKPOINT_DIR, LOG_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("DATA_DIR      :", DATA_DIR)
print("TOKENIZER     :", TOKENIZER_PATH)
print("BASE_DIR      :", BASE_DIR)
print("Dataset train :", len(list(DATA_DIR.glob("train_*.txt"))))
print("Dataset val   :", len(list(DATA_DIR.glob("val_*.txt"))))


DATA_DIR      : /kaggle/input/datasets/vidurshah04/assam-data
TOKENIZER     : /kaggle/working/assamese_10k.model
BASE_DIR      : /kaggle/working/assamese_10k
Dataset train : 10
Dataset val   : 2


In [2]:
%%writefile /kaggle/working/config.py

from dataclasses import dataclass


@dataclass
class ModelConfig:

    # --------------------------------------------------------
    # Vocabulary
    # --------------------------------------------------------

    vocab_size: int = 10_000

    # --------------------------------------------------------
    # Transformer dimensions
    # --------------------------------------------------------

    d_model: int = 512

    n_layers: int = 7

    n_heads: int = 8

    d_ff: int = 1792

    # --------------------------------------------------------
    # Context
    # --------------------------------------------------------

    context_length: int = 512

    # --------------------------------------------------------
    # Regularization
    # --------------------------------------------------------

    dropout: float = 0.1

    @property
    def head_dim(self):

        return self.d_model // self.n_heads

Writing /kaggle/working/config.py


In [3]:
%%writefile /kaggle/working/transformer.py

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


try:
    from .config import ModelConfig
except ImportError:
    from config import ModelConfig


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        assert config.d_model % config.n_heads == 0

        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.q_proj = nn.Linear(config.d_model, config.d_model)
        self.k_proj = nn.Linear(config.d_model, config.d_model)
        self.v_proj = nn.Linear(config.d_model, config.d_model)

        self.out_proj = nn.Linear(config.d_model, config.d_model)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        mask = torch.triu(
            torch.ones(
                config.context_length,
                config.context_length,
                dtype=torch.bool,
            ),
            diagonal=1,
        )

        self.register_buffer("causal_mask", mask)

    def forward(self, x, return_attention=False):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T, self.n_heads, self.head_dim
        ).transpose(1, 2)

        scores = (
            q @ k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        scores = scores.masked_fill(
            self.causal_mask[:T, :T],
            float("-inf"),
        )

        attention = F.softmax(scores, dim=-1)
        attention_for_analysis = attention

        attention = self.attn_dropout(attention)

        out = attention @ v

        out = (
            out.transpose(1, 2)
            .contiguous()
            .view(B, T, C)
        )

        out = self.out_proj(out)
        out = self.resid_dropout(out)

        if return_attention:
            return out, attention_for_analysis

        return out


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff),
            nn.GELU(),
            nn.Linear(config.d_ff, config.d_model),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.ln1 = nn.LayerNorm(config.d_model)
        self.attention = CausalSelfAttention(config)

        self.ln2 = nn.LayerNorm(config.d_model)
        self.ffn = FeedForward(config)

    def forward(self, x, return_attention=False):
        if return_attention:
            attention_output, attention_weights = self.attention(
                self.ln1(x),
                return_attention=True,
            )

            x = x + attention_output
            x = x + self.ffn(self.ln2(x))

            return x, attention_weights

        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))

        return x


class DecoderOnlyTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        # Ablation: token embeddings only.
        # Positional embeddings are deliberately removed.
        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.d_model,
        )

        self.embedding_dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config)
                for _ in range(config.n_layers)
            ]
        )

        self.final_ln = nn.LayerNorm(config.d_model)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.02,
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.Embedding):
                nn.init.normal_(
                    module.weight,
                    mean=0.0,
                    std=0.02,
                )

    def forward(
        self,
        input_ids,
        targets=None,
        return_attention=False,
    ):
        B, T = input_ids.shape

        if T > self.config.context_length:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"context length {self.config.context_length}"
            )

        # IMPORTANT:
        # No positional information is added here.
        x = self.token_embedding(input_ids)

        x = self.embedding_dropout(x)

        attention_maps = []

        for block in self.blocks:
            if return_attention:
                x, attention_weights = block(
                    x,
                    return_attention=True,
                )
                attention_maps.append(attention_weights)
            else:
                x = block(x)

        x = self.final_ln(x)

        logits = F.linear(
            x,
            self.token_embedding.weight,
        )

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
            )

        if return_attention:
            return logits, loss, attention_maps

        return logits, loss

    def count_parameters(self):
        return sum(
            parameter.numel()
            for parameter in self.parameters()
            if parameter.requires_grad
        )


Writing /kaggle/working/transformer.py


In [4]:
import sentencepiece as spm
from pathlib import Path

tokenizer_path = Path("/kaggle/working/assamese_10k.model")
sp = spm.SentencePieceProcessor(model_file=str(tokenizer_path))

print("=" * 70)
print("TOKENIZER VERIFICATION")
print("=" * 70)
print("Path:", tokenizer_path)
print("Vocabulary size:", sp.get_piece_size())

assert sp.get_piece_size() == 10_000
print("PASS: 10K Assamese tokenizer loaded.")


TOKENIZER VERIFICATION
Path: /kaggle/working/assamese_10k.model
Vocabulary size: 10000
PASS: 10K Assamese tokenizer loaded.


In [5]:
%%writefile /kaggle/working/dataset.py

from pathlib import Path

import torch
from torch.utils.data import IterableDataset
import sentencepiece as spm


class TokenStreamDataset(IterableDataset):

    def __init__(
        self,
        data_dir,
        tokenizer_path,
        seq_len=512,
        split="train",
        start_file=0,
        start_line=0,
        start_buffer=None,
    ):

        self.data_dir = Path(data_dir)

        self.seq_len = int(seq_len)

        self.split = split

        self.tokenizer = (
            spm.SentencePieceProcessor(
                model_file=str(tokenizer_path)
            )
        )

        self.files = sorted(
            self.data_dir.glob(
                f"{split}_*.txt"
            )
        )

        if not self.files:

            raise FileNotFoundError(
                f"No {split}_*.txt files found in "
                f"{self.data_dir}"
            )

        # ----------------------------------------------------
        # Resume state
        # ----------------------------------------------------

        self.start_file = int(
            start_file
        )

        self.start_line = int(
            start_line
        )

        self.start_buffer = (
            list(start_buffer)
            if start_buffer is not None
            else []
        )

        # Runtime state

        self.current_file_index = (
            self.start_file
        )

        self.current_line_number = (
            self.start_line
        )

        self.current_buffer = list(
            self.start_buffer
        )

        print(
            f"{split}: found "
            f"{len(self.files)} files"
        )

        print(
            f"{split}: tokenizer vocabulary size = "
            f"{self.tokenizer.get_piece_size()}"
        )


class ResumableTokenStreamDataset(
    TokenStreamDataset
):

    def __iter__(self):

        token_buffer = list(
            self.start_buffer
        )

        for file_index in range(
            self.start_file,
            len(self.files),
        ):

            self.current_file_index = (
                file_index
            )

            file_path = self.files[
                file_index
            ]

            print(
                f"Reading {self.split}: "
                f"{file_path.name}"
            )

            line_start = (

                self.start_line

                if file_index == self.start_file

                else 0
            )

            with open(
                file_path,
                "r",
                encoding="utf-8",
            ) as f:

                for line_number, line in enumerate(
                    f
                ):

                    if line_number < line_start:

                        continue

                    line = line.strip()

                    if not line:

                        continue

                    tokens = (
                        self.tokenizer.encode(
                            line,
                            out_type=int,
                        )
                    )

                    token_buffer.extend(
                        tokens
                    )

                    while len(
                        token_buffer
                    ) >= self.seq_len:

                        chunk = token_buffer[
                            :self.seq_len
                        ]

                        token_buffer = (
                            token_buffer[
                                self.seq_len:
                            ]
                        )

                        # Save state after
                        # this sequence

                        self.current_file_index = (
                            file_index
                        )

                        self.current_line_number = (
                            line_number + 1
                        )

                        self.current_buffer = list(
                            token_buffer
                        )

                        # Causal LM shift

                        x = torch.tensor(
                            chunk[:-1],
                            dtype=torch.long,
                        )

                        y = torch.tensor(
                            chunk[1:],
                            dtype=torch.long,
                        )

                        yield x, y

            # Move to next file

            self.current_file_index = (
                file_index + 1
            )

            self.current_line_number = 0

            self.current_buffer = list(
                token_buffer
            )

        self.start_file = (
            self.current_file_index
        )

        self.start_line = (
            self.current_line_number
        )

        self.start_buffer = list(
            self.current_buffer
        )

    def get_state(self):

        return {

            "file_index": int(
                getattr(
                    self,
                    "current_file_index",
                    self.start_file,
                )
            ),

            "line_number": int(
                getattr(
                    self,
                    "current_line_number",
                    self.start_line,
                )
            ),

            "token_buffer": list(
                getattr(
                    self,
                    "current_buffer",
                    self.start_buffer,
                )
            ),
        }

    def state_dict(self):

        return self.get_state()

    def load_state_dict(
        self,
        state,
    ):

        self.start_file = int(
            state["file_index"]
        )

        self.start_line = int(
            state["line_number"]
        )

        self.start_buffer = list(
            state["token_buffer"]
        )

        self.current_file_index = (
            self.start_file
        )

        self.current_line_number = (
            self.start_line
        )

        self.current_buffer = list(
            self.start_buffer
        )

        print(
            "Dataset state restored:"
        )

        print(
            f"  file   = {self.start_file}"
        )

        print(
            f"  line   = {self.start_line}"
        )

        print(
            f"  buffer = "
            f"{len(self.start_buffer)} tokens"
        )

Writing /kaggle/working/dataset.py


In [6]:
import torch
import sys
from pathlib import Path

sys.path.insert(0, "/kaggle/working")

from config import ModelConfig
from transformer import DecoderOnlyTransformer

print("=" * 70)
print("POSITIONAL-EMBEDDING ABLATION ARCHITECTURE CHECK")
print("=" * 70)

config = ModelConfig()
model = DecoderOnlyTransformer(config)

params = model.count_parameters()

print("Vocabulary:", config.vocab_size)
print("d_model  :", config.d_model)
print("Layers   :", config.n_layers)
print("Heads    :", config.n_heads)
print("FFN      :", config.d_ff)
print("Context  :", config.context_length)
print("Params   :", f"{params:,}")

assert config.vocab_size == 10_000
assert not hasattr(model, "position_embedding")
assert params == 25_350_912

print("Positional embeddings: REMOVED")
print("Parameter count       :", params)
print("Architecture check    : PASS")


POSITIONAL-EMBEDDING ABLATION ARCHITECTURE CHECK
Vocabulary: 10000
d_model  : 512
Layers   : 7
Heads    : 8
FFN      : 1792
Context  : 512
Params   : 25,350,912
Positional embeddings: REMOVED
Parameter count       : 25350912
Architecture check    : PASS


In [7]:
import torch

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

assert torch.cuda.is_available(), "CUDA GPU is required."

print("CUDA available:", torch.cuda.is_available())
print("GPU            :", torch.cuda.get_device_name(0))
print("CUDA version   :", torch.version.cuda)
print(
    "GPU memory     :",
    round(
        torch.cuda.get_device_properties(0).total_memory / (1024 ** 3),
        2,
    ),
    "GB",
)


GPU CHECK
CUDA available: True
GPU            : Tesla T4
CUDA version   : 12.8
GPU memory     : 14.56 GB


In [8]:
from pathlib import Path

data_dir = Path("/kaggle/input/datasets/vidurshah04/assam-data")
if not data_dir.exists():
    data_dir = Path("/kaggle/input/assam-data")

train_files = sorted(data_dir.glob("train_*.txt"))
val_files = sorted(data_dir.glob("val_*.txt"))

assert train_files, f"No train_*.txt files in {data_dir}"
assert val_files, f"No val_*.txt files in {data_dir}"

print("=" * 70)
print("DATASET CHECK")
print("=" * 70)
print("Data directory:", data_dir)
print("Training files:", len(train_files))
print("Validation files:", len(val_files))

print("PASS: Assamese dataset is visible.")


DATASET CHECK
Data directory: /kaggle/input/datasets/vidurshah04/assam-data
Training files: 10
Validation files: 2
PASS: Assamese dataset is visible.


In [9]:
%%writefile /kaggle/working/train_assamese_10k.py

from pathlib import Path
import math
import time
import csv
import os

import torch
from torch.utils.data import DataLoader

from config import ModelConfig
from transformer import DecoderOnlyTransformer
from dataset import ResumableTokenStreamDataset


# ============================================================
# PATHS
# ============================================================

DATA_DIR = Path(
    os.environ.get(
        "ABLATION_DATA_DIR",
        "/kaggle/input/datasets/vidurshah04/assam-data",
    )
)

TOKENIZER_PATH = Path(
    "/kaggle/working/assamese_10k.model"
)

BASE_DIR = Path(
    "/kaggle/working/assamese_10k"
)

CHECKPOINT_DIR = (
    BASE_DIR / "checkpoints"
)

LOG_DIR = (
    BASE_DIR / "logs"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# RESUME
# ============================================================

# Automatically resume from the latest valid checkpoint if one
# already exists. On a first run there will be none.
AUTO_RESUME = True
RESUME_CHECKPOINT = None

if AUTO_RESUME:
    existing_checkpoints = sorted(
        CHECKPOINT_DIR.glob("checkpoint_*.pt"),
        key=lambda p: int(p.stem.split("_")[1]),
    )
    if existing_checkpoints:
        RESUME_CHECKPOINT = existing_checkpoints[-1]
        print("AUTO-RESUME checkpoint:", RESUME_CHECKPOINT)


# ============================================================
# TRAINING CONFIG
# ============================================================

BATCH_SIZE = 1

GRAD_ACCUMULATION = 32

SEQ_LEN = 512

TARGET_TOKENS = 500_000_000

TOKENS_PER_OPTIMIZER_STEP = (
    BATCH_SIZE
    * GRAD_ACCUMULATION
    * SEQ_LEN
)

MAX_STEPS = math.ceil(
    TARGET_TOKENS
    / TOKENS_PER_OPTIMIZER_STEP
)

LEARNING_RATE = 3e-4

MIN_LR = 3e-5

WEIGHT_DECAY = 0.1

BETAS = (
    0.9,
    0.95,
)

WARMUP_STEPS = 2_000

VALIDATE_EVERY = 1_000

SAVE_EVERY = 3_000

VAL_BATCHES = 100

GRAD_CLIP = 1.0


# ============================================================
# DEVICE
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is not available."
    )

device = torch.device(
    "cuda"
)


# ============================================================
# MODEL
# ============================================================

config = ModelConfig()

model = DecoderOnlyTransformer(
    config
).to(device)

parameter_count = (
    model.count_parameters()
)

print("=" * 70)
print("assamese POSITIONAL-EMBEDDING ABLATION")
print("=" * 70)

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "Vocabulary:",
    config.vocab_size
)

print(
    "Parameters:",
    f"{parameter_count:,}"
)

assert config.vocab_size == 10_000

assert parameter_count >= 25_000_000


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=BETAS,
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# AMP
# ============================================================

USE_AMP = True

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)


# ============================================================
# LR SCHEDULE
# ============================================================

def get_lr(step):

    if step < WARMUP_STEPS:

        return (
            LEARNING_RATE
            * (step + 1)
            / WARMUP_STEPS
        )

    progress = (
        step - WARMUP_STEPS
    ) / (
        MAX_STEPS - WARMUP_STEPS
    )

    progress = min(
        max(progress, 0.0),
        1.0,
    )

    return MIN_LR + (
        LEARNING_RATE - MIN_LR
    ) * 0.5 * (
        1.0
        + math.cos(
            math.pi * progress
        )
    )


# ============================================================
# RESUME STATE
# ============================================================

dataset_state = {

    "file_index": 0,

    "line_number": 0,

    "token_buffer": [],
}

start_step = 0

last_loss = 0.0

last_val_loss = None


# ============================================================
# LOAD CHECKPOINT
# ============================================================

if RESUME_CHECKPOINT is not None:

    print("=" * 70)
    print("RESUMING TRAINING")
    print("=" * 70)

    if not RESUME_CHECKPOINT.exists():

        raise FileNotFoundError(
            RESUME_CHECKPOINT
        )

    checkpoint = torch.load(
        RESUME_CHECKPOINT,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )

    if (
        "scaler_state_dict"
        in checkpoint
    ):

        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )

    start_step = int(
        checkpoint["step"]
    )

    dataset_state = checkpoint[
        "dataset_state"
    ]

    last_loss = checkpoint.get(
        "loss",
        0.0,
    )

    last_val_loss = checkpoint.get(
        "val_loss",
        None,
    )

    restored_lr = checkpoint.get(
        "learning_rate",
        get_lr(start_step),
    )

    for group in optimizer.param_groups:

        group["lr"] = restored_lr

    print(
        "Resumed from:",
        start_step
    )

    print(
        "Previous loss:",
        last_loss
    )


# ============================================================
# LOSS CSV
# ============================================================

LOSS_CSV = (
    LOG_DIR / "training_log.csv"
)


if not LOSS_CSV.exists():

    with open(
        LOSS_CSV,
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.writer(f)

        writer.writerow(
            [
                "step",
                "train_loss",
                "val_loss",
                "learning_rate",
                "tokens",
                "elapsed_hours",
            ]
        )


def write_log(
    step,
    train_loss,
    val_loss,
    learning_rate,
    elapsed_hours,
):

    with open(
        LOSS_CSV,
        "a",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.writer(f)

        writer.writerow(
            [
                step,
                train_loss,
                val_loss,
                learning_rate,
                step * TOKENS_PER_OPTIMIZER_STEP,
                elapsed_hours,
            ]
        )


# ============================================================
# TRAIN DATASET
# ============================================================

print("=" * 70)
print("TRAIN DATASET")
print("=" * 70)

train_dataset = (
    ResumableTokenStreamDataset(

        data_dir=DATA_DIR,

        tokenizer_path=TOKENIZER_PATH,

        seq_len=SEQ_LEN,

        split="train",

        start_file=dataset_state[
            "file_index"
        ],

        start_line=dataset_state[
            "line_number"
        ],

        start_buffer=dataset_state[
            "token_buffer"
        ],
    )
)

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    num_workers=0,

    pin_memory=True,
)

train_iterator = iter(
    train_loader
)


# ============================================================
# VALIDATION
# ============================================================

def validate():

    model.eval()

    val_dataset = (
        ResumableTokenStreamDataset(

            data_dir=DATA_DIR,

            tokenizer_path=TOKENIZER_PATH,

            seq_len=SEQ_LEN,

            split="val",
        )
    )

    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        num_workers=0,

        pin_memory=True,
    )

    val_iterator = iter(
        val_loader
    )

    losses = []

    with torch.no_grad():

        for _ in range(
            VAL_BATCHES
        ):

            try:

                x, y = next(
                    val_iterator
                )

            except StopIteration:

                break

            x = x.to(
                device,
                non_blocking=True,
            )

            y = y.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP,
            ):

                _, loss = model(
                    x,
                    y,
                )

            if torch.isfinite(loss):

                losses.append(
                    loss.item()
                )

    model.train()

    if not losses:

        return float("nan")

    val_loss = (
        sum(losses)
        / len(losses)
    )

    ppl = math.exp(
        min(val_loss, 20)
    )

    print(
        f"Validation loss: "
        f"{val_loss:.4f} | "
        f"PPL: {ppl:.2f}"
    )

    return val_loss


# ============================================================
# CHECKPOINT
# ============================================================

def save_checkpoint(
    step,
    loss,
    val_loss,
):

    print("\n" + "=" * 70)

    print(
        f"SAVING CHECKPOINT "
        f"{step:06d}"
    )

    print("=" * 70)

    checkpoint = {

        "step":
            int(step),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "config":
            vars(config),

        "loss":
            float(loss),

        "val_loss":
            (
                None
                if val_loss is None
                else float(val_loss)
            ),

        "learning_rate":
            float(get_lr(step)),

        "dataset_state":
            train_dataset.get_state(),
    }

    local_path = (
        CHECKPOINT_DIR
        / f"checkpoint_{step:06d}.pt"
    )

    temp_path = (
        CHECKPOINT_DIR
        / f".checkpoint_{step:06d}.tmp"
    )

    torch.save(
        checkpoint,
        temp_path,
    )

    temp_path.replace(
        local_path
    )

    print(
        "Checkpoint saved:",
        local_path
    )

    print(
        "Checkpoint size:",
        f"{local_path.stat().st_size / (1024**2):.1f} MB"
    )


# ============================================================
# TRAINING
# ============================================================

print("\n" + "=" * 70)
print("TRAINING START")
print("=" * 70)

print(
    "Starting step:",
    start_step + 1
)

print(
    "Final step:",
    MAX_STEPS
)

print(
    "Target tokens:",
    f"{TARGET_TOKENS:,}"
)

print(
    "Tokens / optimizer step:",
    f"{TOKENS_PER_OPTIMIZER_STEP:,}"
)

print(
    "Loss CSV:",
    LOSS_CSV
)

print()

start_time = time.time()

interval_start = time.time()

model.train()


for step in range(
    start_step + 1,
    MAX_STEPS + 1,
):

    optimizer.zero_grad(
        set_to_none=True
    )

    total_loss = 0.0

    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------

    for microstep in range(
        GRAD_ACCUMULATION
    ):

        x, y = next(
            train_iterator
        )

        x = x.to(
            device,
            non_blocking=True,
        )

        y = y.to(
            device,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP,
        ):

            _, loss = model(
                x,
                y,
            )

            loss = (
                loss
                / GRAD_ACCUMULATION
            )

        scaler.scale(
            loss
        ).backward()

        total_loss += loss.item()

    # --------------------------------------------------------
    # Gradient clipping
    # --------------------------------------------------------

    scaler.unscale_(
        optimizer
    )

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        GRAD_CLIP,
    )

    # --------------------------------------------------------
    # Learning rate
    # --------------------------------------------------------

    lr = get_lr(step)

    for group in optimizer.param_groups:

        group["lr"] = lr

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    scaler.step(
        optimizer
    )

    scaler.update()

    last_loss = total_loss

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    if step % 100 == 0:

        elapsed_hours = (
            time.time()
            - start_time
        ) / 3600

        interval_time = (
            time.time()
            - interval_start
        )

        print(
            f"step={step} | "
            f"loss={last_loss:.4f} | "
            f"lr={lr:.6e} | "
            f"tokens="
            f"{step * TOKENS_PER_OPTIMIZER_STEP:,} | "
            f"time={interval_time:.1f}s/100 | "
            f"total={elapsed_hours:.2f}h"
        )

        write_log(
            step,
            last_loss,
            last_val_loss,
            lr,
            elapsed_hours,
        )

        interval_start = time.time()

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    if step % VALIDATE_EVERY == 0:

        last_val_loss = validate()

        elapsed_hours = (
            time.time()
            - start_time
        ) / 3600

        write_log(
            step,
            last_loss,
            last_val_loss,
            lr,
            elapsed_hours,
        )

    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    if step % SAVE_EVERY == 0:

        save_checkpoint(
            step,
            last_loss,
            last_val_loss,
        )


# ============================================================
# FINAL CHECKPOINT
# ============================================================

print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

if MAX_STEPS % SAVE_EVERY != 0:

    save_checkpoint(
        MAX_STEPS,
        last_loss,
        last_val_loss,
    )

print(
    "Final step:",
    MAX_STEPS
)

print(
    "Final tokens:",
    f"{MAX_STEPS * TOKENS_PER_OPTIMIZER_STEP:,}"
)

print(
    "Loss log:",
    LOSS_CSV
)

print(
    "Total time:",
    f"{(time.time() - start_time) / 3600:.2f} hours"
)

Writing /kaggle/working/train_assamese_10k.py


In [10]:
from pathlib import Path
import py_compile

script = Path("/kaggle/working/train_assamese_10k.py")
py_compile.compile(str(script), doraise=True)

print("=" * 70)
print("TRAINING SCRIPT CHECK")
print("=" * 70)
print("Script:", script)
print("Size  :", f"{script.stat().st_size / 1024:.1f} KB")
print("PASS  : Python syntax compilation")


TRAINING SCRIPT CHECK
Script: /kaggle/working/train_assamese_10k.py
Size  : 14.7 KB
PASS  : Python syntax compilation


In [11]:
import subprocess
import sys
import os
from pathlib import Path

train_script = Path("/kaggle/working/train_assamese_10k.py")

print("=" * 80)
print("STARTING REMOTE TRAINING")
print("=" * 80)
print("This cell may run for several hours.")
print("The training script saves checkpoints every 3,000 steps and")
print("also writes a final checkpoint at step 30,518.")
print()

env = os.environ.copy()
env["ABLATION_DATA_DIR"] = str(DATA_DIR)

subprocess.run(
    [sys.executable, "-u", str(train_script)],
    env=env,
    check=True,
)

print()
print("=" * 80)
print("TRAINING PROCESS RETURNED SUCCESSFULLY")
print("=" * 80)


STARTING REMOTE TRAINING
This cell may run for several hours.
The training script saves checkpoints every 3,000 steps and
also writes a final checkpoint at step 30,518.

assamese POSITIONAL-EMBEDDING ABLATION
GPU: Tesla T4
Vocabulary: 10000
Parameters: 25,350,912
TRAIN DATASET
train: found 10 files
train: tokenizer vocabulary size = 10000

TRAINING START
Starting step: 1
Final step: 30518
Target tokens: 500,000,000
Tokens / optimizer step: 16,384
Loss CSV: /kaggle/working/assamese_10k/logs/training_log.csv

Reading train: train_0000.txt
step=100 | loss=8.6877 | lr=1.515000e-05 | tokens=1,638,400 | time=78.1s/100 | total=0.02h
step=200 | loss=8.1100 | lr=3.015000e-05 | tokens=3,276,800 | time=75.4s/100 | total=0.04h
step=300 | loss=7.4227 | lr=4.515000e-05 | tokens=4,915,200 | time=75.0s/100 | total=0.06h
step=400 | loss=6.3712 | lr=6.015000e-05 | tokens=6,553,600 | time=75.2s/100 | total=0.08h
step=500 | loss=6.2883 | lr=7.515000e-05 | tokens=8,192,000 | time=75.2s/100 | total=0.11h
st

In [12]:
%%writefile /kaggle/working/ablation_standard_generation.py
from pathlib import Path
import os
import sys
import math
import random

import sentencepiece as spm
import torch


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

PROJECT_ROOT = Path("/kaggle/working")
MODEL_DIR = PROJECT_ROOT

CHECKPOINT_PATH = Path(os.environ["ABLATION_CHECKPOINT"])

TOKENIZER_PATH = Path("/kaggle/working/assamese_10k.model")

VAL_DIR = Path("/kaggle/input/datasets/vidurshah04/assam-data")

OUTPUT_DIR = Path(os.environ["ABLATION_EVAL_DIR"]) / "generations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(MODEL_DIR))

from config import ModelConfig
from transformer import DecoderOnlyTransformer


# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------

CONTEXT_LENGTH = 256
MAX_NEW_TOKENS = 100

# TEMPERATURES = [
#     0.5,
#     1.0,
#     1.5,
# ]
TEMPERATURES = [
    0.0,   # greedy
    0.5,
    1.0,
    1.5,
]

NUM_PROMPTS = 10

SEED = 42


# ---------------------------------------------------------
# Load validation prompts
# ---------------------------------------------------------

def load_prompts():

    prompts = []

    validation_files = sorted(
        VAL_DIR.glob("val_*.txt")
    )

    if not validation_files:
        raise FileNotFoundError(
            f"No validation files found in {VAL_DIR}"
        )

    for file_path in validation_files:

        with open(
            file_path,
            "r",
            encoding="utf-8",
        ) as f:

            for line in f:

                line = line.strip()

                # We want reasonably long natural-language
                # examples rather than tiny fragments.
                if len(line) < 100:
                    continue

                prompts.append(line)

                if len(prompts) >= NUM_PROMPTS:
                    return prompts

    if len(prompts) < NUM_PROMPTS:
        raise RuntimeError(
            f"Only found {len(prompts)} suitable prompts."
        )

    return prompts


# ---------------------------------------------------------
# Generation
# ---------------------------------------------------------

@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt,
    device,
    temperature,
    max_new_tokens,
):

    encoded = tokenizer.encode(
        prompt,
        out_type=int,
    )

    # Keep only the last CONTEXT_LENGTH tokens.
    encoded = encoded[-CONTEXT_LENGTH:]

    input_ids = torch.tensor(
        [encoded],
        dtype=torch.long,
        device=device,
    )

    generated = input_ids.clone()

    for _ in range(max_new_tokens):

        # Transformer only accepts at most its context length.
        context = generated[:, -CONTEXT_LENGTH:]

        logits, _ = model(
            context
        )

        # Only predict the next token from the
        # final position.
        next_token_logits = logits[:, -1, :]

        if temperature <= 0:
            next_token = torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True,
            )

        else:
            scaled_logits = (
                next_token_logits / temperature
            )

            probabilities = torch.softmax(
                scaled_logits,
                dim=-1,
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1,
            )

        generated = torch.cat(
            [generated, next_token],
            dim=1,
        )

    generated_ids = generated[0].tolist()

    return tokenizer.decode(
        generated_ids
    )


# ---------------------------------------------------------
# Main
# ---------------------------------------------------------

def main():

    print("=" * 70)
    print("ASSAMESE GENERATION EVALUATION")
    print("=" * 70)

    random.seed(SEED)
    torch.manual_seed(SEED)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(f"Device: {device}")

    if device.type == "cuda":
        print(
            f"GPU: {torch.cuda.get_device_name(0)}"
        )

    # -----------------------------------------------------
    # Tokenizer
    # -----------------------------------------------------

    tokenizer = spm.SentencePieceProcessor()

    if not tokenizer.load(
        str(TOKENIZER_PATH)
    ):
        raise RuntimeError(
            f"Could not load tokenizer: {TOKENIZER_PATH}"
        )

    print(
        f"Tokenizer vocabulary: "
        f"{tokenizer.get_piece_size()}"
    )

    # -----------------------------------------------------
    # Model
    # -----------------------------------------------------

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=False,
    )

    # config = ModelConfig()
    config = ModelConfig(**checkpoint["config"])

    model = DecoderOnlyTransformer(
        config
    ).to(device)

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.eval()

    print(
        f"Checkpoint step: "
        f"{checkpoint['step']}"
    )

    # -----------------------------------------------------
    # Prompts
    # -----------------------------------------------------

    prompts = load_prompts()

    print(
        f"Using {len(prompts)} validation prompts."
    )

    # -----------------------------------------------------
    # Generate
    # -----------------------------------------------------

    for temperature in TEMPERATURES:

        print(
            f"\nGenerating at temperature "
            f"{temperature}..."
        )

        output_path = (
            OUTPUT_DIR
            / f"assamese_temp_{temperature:.1f}.txt"
        )

        with open(
            output_path,
            "w",
            encoding="utf-8",
        ) as out:

            for i, prompt in enumerate(prompts):

                generated_text = generate(
                    model=model,
                    tokenizer=tokenizer,
                    prompt=prompt,
                    device=device,
                    temperature=temperature,
                    max_new_tokens=MAX_NEW_TOKENS,
                )

                # Store the original prompt separately.
                out.write(
                    f"========== SAMPLE {i + 1} ==========\n"
                )

                out.write(
                    "PROMPT:\n"
                )

                out.write(
                    prompt + "\n\n"
                )

                out.write(
                    "GENERATED:\n"
                )

                out.write(
                    generated_text + "\n\n"
                )

        print(
            f"Saved: {output_path}"
        )

    print("\nGeneration complete.")


if __name__ == "__main__":
    main()

Writing /kaggle/working/ablation_standard_generation.py


In [13]:
%%writefile /kaggle/working/ablation_generation_metrics.py
from pathlib import Path
import os
import sys
import json
import math
import random
import re

# Add Assamese/ to Python's import path
PROJECT_DIR = Path("/kaggle/working")
sys.path.insert(0, str(PROJECT_DIR))

import torch
import sentencepiece as spm

from config import ModelConfig
from transformer import DecoderOnlyTransformer


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

NUM_SAMPLES = 50
PROMPT_TOKENS = 128
REFERENCE_TOKENS = 100
MAX_NEW_TOKENS = 100

TEMPERATURES = [0.0, 0.5, 1.0, 1.5]

DATA_DIR = Path("/kaggle/input/datasets/vidurshah04/assam-data")

TOKENIZER_PATH = Path("/kaggle/working/assamese_10k.model")

CHECKPOINT_PATH = Path(os.environ["ABLATION_CHECKPOINT"])

RESULTS_DIR = Path(os.environ["ABLATION_EVAL_DIR"]) / "generation_metrics"

GENERATIONS_DIR = Path(os.environ["ABLATION_EVAL_DIR"]) / "generation_metrics_data"


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# TOKENIZER
# ============================================================

sp = spm.SentencePieceProcessor()

if not sp.load(str(TOKENIZER_PATH)):
    raise RuntimeError(f"Could not load tokenizer: {TOKENIZER_PATH}")


# ============================================================
# MODEL
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

config_dict = checkpoint["config"]

config = ModelConfig(**config_dict)

model = DecoderOnlyTransformer(config).to(device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("=" * 70)
print("assamese GENERATION METRICS")
print("=" * 70)
print(f"Device: {device}")
print(f"Checkpoint step: {checkpoint['step']}")
print(f"Vocabulary: {sp.get_piece_size()}")
print(f"Samples: {NUM_SAMPLES}")
print(f"Prompt tokens: {PROMPT_TOKENS}")
print(f"Reference tokens: {REFERENCE_TOKENS}")
print()


# ============================================================
# LOAD VALIDATION EXAMPLES
# ============================================================

candidate_examples = []

files = sorted(DATA_DIR.glob("val_*.txt"))

for file_path in files:

    with open(file_path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            ids = sp.encode(line, out_type=int)

            required = PROMPT_TOKENS + REFERENCE_TOKENS

            if len(ids) >= required:
                candidate_examples.append(ids)

            if len(candidate_examples) >= NUM_SAMPLES * 5:
                break

        if len(candidate_examples) >= NUM_SAMPLES * 5:
            break


if len(candidate_examples) < NUM_SAMPLES:
    raise RuntimeError(
        f"Only found {len(candidate_examples)} suitable validation examples."
    )


# Deterministically select examples
random.shuffle(candidate_examples)
examples = candidate_examples[:NUM_SAMPLES]


# ============================================================
# GENERATION
# ============================================================

@torch.no_grad()
def generate_continuation(prompt_ids, temperature):

    generated = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device,
    )

    for _ in range(MAX_NEW_TOKENS):

        # Keep only the model's maximum context
        context = generated[:, -config.context_length:]

        logits, _ = model(context)

        next_logits = logits[:, -1, :]

        if temperature == 0.0:
            next_token = torch.argmax(
                next_logits,
                dim=-1,
                keepdim=True,
            )

        else:
            scaled_logits = next_logits / temperature

            probabilities = torch.softmax(
                scaled_logits,
                dim=-1,
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1,
            )

        generated = torch.cat(
            [generated, next_token],
            dim=1,
        )

    return generated[0, -MAX_NEW_TOKENS:].tolist()


# ============================================================
# TEXT NORMALIZATION
# ============================================================

def normalize_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


def word_tokens(text):
    return normalize_text(text).split()


def character_tokens(text):
    return list(normalize_text(text))


# ============================================================
# BLEU-4
# ============================================================

def ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


def modified_precision(reference, candidate, n):

    ref_counts = {}

    for gram in ngrams(reference, n):
        ref_counts[gram] = ref_counts.get(gram, 0) + 1

    cand_counts = {}

    for gram in ngrams(candidate, n):
        cand_counts[gram] = cand_counts.get(gram, 0) + 1

    clipped = 0

    for gram, count in cand_counts.items():
        clipped += min(
            count,
            ref_counts.get(gram, 0)
        )

    total = max(len(candidate) - n + 1, 0)

    return clipped, total


def bleu4(reference, candidate):

    precisions = []

    for n in range(1, 5):

        clipped, total = modified_precision(
            reference,
            candidate,
            n,
        )

        # Small smoothing so short generations don't
        # automatically receive BLEU = 0.
        precision = (clipped + 1) / (total + 1)

        precisions.append(precision)

    if not candidate:
        return 0.0

    log_precision = sum(
        math.log(p)
        for p in precisions
    ) / 4.0

    geometric_mean = math.exp(log_precision)

    brevity_penalty = 1.0

    if len(candidate) < len(reference):
        brevity_penalty = math.exp(
            1.0 - len(reference) / max(len(candidate), 1)
        )

    return brevity_penalty * geometric_mean


# ============================================================
# CHRF
# ============================================================

def chrf(reference, candidate, max_order=6, beta=2):

    scores = []

    for n in range(1, max_order + 1):

        ref_ngrams = ngrams(
            character_tokens(reference),
            n,
        )

        cand_ngrams = ngrams(
            character_tokens(candidate),
            n,
        )

        ref_counts = {}
        cand_counts = {}

        for gram in ref_ngrams:
            ref_counts[gram] = ref_counts.get(gram, 0) + 1

        for gram in cand_ngrams:
            cand_counts[gram] = cand_counts.get(gram, 0) + 1

        overlap = sum(
            min(count, ref_counts.get(gram, 0))
            for gram, count in cand_counts.items()
        )

        precision = overlap / max(len(cand_ngrams), 1)
        recall = overlap / max(len(ref_ngrams), 1)

        if precision + recall == 0:
            f_score = 0.0
        else:
            f_score = (
                (1 + beta ** 2)
                * precision
                * recall
                / (beta ** 2 * precision + recall)
            )

        scores.append(f_score)

    return sum(scores) / len(scores)


# ============================================================
# ROUGE-L
# ============================================================

def lcs_length(a, b):

    previous = [0] * (len(b) + 1)

    for token_a in a:

        current = [0]

        for j, token_b in enumerate(b, start=1):

            if token_a == token_b:
                current.append(previous[j - 1] + 1)
            else:
                current.append(
                    max(previous[j], current[-1])
                )

        previous = current

    return previous[-1]


def rouge_l(reference, candidate):

    ref = word_tokens(reference)
    cand = word_tokens(candidate)

    if not ref or not cand:
        return 0.0

    lcs = lcs_length(ref, cand)

    precision = lcs / len(cand)
    recall = lcs / len(ref)

    if precision + recall == 0:
        return 0.0

    beta = 1.2

    return (
        (1 + beta ** 2)
        * precision
        * recall
        / (recall + beta ** 2 * precision)
    )


# ============================================================
# DIVERSITY METRICS
# ============================================================

def distinct_n(text, n):

    tokens = word_tokens(text)

    if len(tokens) < n:
        return 0.0

    grams = ngrams(tokens, n)

    return len(set(grams)) / len(grams)


def repetition_rate(text):

    tokens = word_tokens(text)

    if len(tokens) < 2:
        return 0.0

    grams = ngrams(tokens, 2)

    seen = set()
    repeated = 0

    for gram in grams:

        if gram in seen:
            repeated += 1

        seen.add(gram)

    return repeated / len(grams)


# ============================================================
# RUN EVALUATION
# ============================================================

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GENERATIONS_DIR.mkdir(parents=True, exist_ok=True)

all_results = []


for temperature in TEMPERATURES:

    print(f"Generating/evaluating T={temperature}...")

    temperature_results = []

    for index, ids in enumerate(examples):

        prompt_ids = ids[:PROMPT_TOKENS]
        reference_ids = ids[
            PROMPT_TOKENS:
            PROMPT_TOKENS + REFERENCE_TOKENS
        ]

        generated_ids = generate_continuation(
            prompt_ids,
            temperature,
        )

        reference_text = normalize_text(
            sp.decode(reference_ids)
        )

        generated_text = normalize_text(
            sp.decode(generated_ids)
        )

        result = {
            "sample": index,
            "temperature": temperature,
            "reference": reference_text,
            "generated": generated_text,
            "bleu4": bleu4(
                word_tokens(reference_text),
                word_tokens(generated_text),
            ),
            "chrf": chrf(
                reference_text,
                generated_text,
            ),
            "rouge_l": rouge_l(
                reference_text,
                generated_text,
            ),
            "repetition_rate": repetition_rate(
                generated_text
            ),
            "distinct_1": distinct_n(
                generated_text,
                1,
            ),
            "distinct_2": distinct_n(
                generated_text,
                2,
            ),
        }

        temperature_results.append(result)
        all_results.append(result)

    # Save human-readable generations
    output_file = (
        GENERATIONS_DIR
        / f"assamese_metrics_temp_{temperature}.json"
    )

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            temperature_results,
            f,
            ensure_ascii=False,
            indent=2,
        )


# ============================================================
# AGGREGATE RESULTS
# ============================================================

summary = []

for temperature in TEMPERATURES:

    rows = [
        r for r in all_results
        if r["temperature"] == temperature
    ]

    summary.append({
        "temperature": temperature,
        "samples": len(rows),
        "BLEU-4": sum(r["bleu4"] for r in rows) / len(rows),
        "chrF": sum(r["chrf"] for r in rows) / len(rows),
        "ROUGE-L": sum(r["rouge_l"] for r in rows) / len(rows),
        "Repetition Rate": sum(
            r["repetition_rate"] for r in rows
        ) / len(rows),
        "Distinct-1": sum(
            r["distinct_1"] for r in rows
        ) / len(rows),
        "Distinct-2": sum(
            r["distinct_2"] for r in rows
        ) / len(rows),
    })


# ============================================================
# SAVE SUMMARY
# ============================================================

summary_file = RESULTS_DIR / "assamese_generation_metrics.json"

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )


# CSV
csv_file = RESULTS_DIR / "assamese_generation_metrics.csv"

with open(csv_file, "w", encoding="utf-8") as f:

    headers = [
        "temperature",
        "samples",
        "BLEU-4",
        "chrF",
        "ROUGE-L",
        "Repetition Rate",
        "Distinct-1",
        "Distinct-2",
    ]

    f.write(",".join(headers) + "\n")

    for row in summary:

        f.write(
            f"{row['temperature']},"
            f"{row['samples']},"
            f"{row['BLEU-4']:.6f},"
            f"{row['chrF']:.6f},"
            f"{row['ROUGE-L']:.6f},"
            f"{row['Repetition Rate']:.6f},"
            f"{row['Distinct-1']:.6f},"
            f"{row['Distinct-2']:.6f}\n"
        )


# ============================================================
# PRINT RESULTS
# ============================================================

print()
print("=" * 70)
print("assamese GENERATION METRICS")
print("=" * 70)

for row in summary:

    print(f"\nTemperature: {row['temperature']}")
    print(f"  BLEU-4          : {row['BLEU-4']:.6f}")
    print(f"  chrF            : {row['chrF']:.6f}")
    print(f"  ROUGE-L         : {row['ROUGE-L']:.6f}")
    print(f"  Repetition Rate : {row['Repetition Rate']:.6f}")
    print(f"  Distinct-1      : {row['Distinct-1']:.6f}")
    print(f"  Distinct-2      : {row['Distinct-2']:.6f}")

print()
print(f"Saved summary: {csv_file}")
print(f"Saved JSON:    {summary_file}")
print("=" * 70)

Writing /kaggle/working/ablation_generation_metrics.py


In [14]:
%%writefile /kaggle/working/ablation_attention_aggregate.py
from pathlib import Path
import os
import sys
import csv

import torch
import matplotlib.pyplot as plt
import sentencepiece as spm

# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path("/kaggle/working")
sys.path.insert(0, str(PROJECT_DIR))

from config import ModelConfig
from transformer import DecoderOnlyTransformer


DATA_DIR = Path("/kaggle/input/datasets/vidurshah04/assam-data")

TOKENIZER_PATH = Path("/kaggle/working/assamese_10k.model")

CHECKPOINT_PATH = Path(os.environ["ABLATION_CHECKPOINT"])

RESULTS_DIR = Path(os.environ["ABLATION_EVAL_DIR"]) / "attention"

FIGURES_DIR = Path(os.environ["ABLATION_EVAL_DIR"]) / "attention_figures"

SEQUENCE_LENGTH = 128
NUM_SEQUENCES = 10


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("assamese AGGREGATED ATTENTION ANALYSIS")
print("=" * 70)

print(f"Device: {device}")


# ============================================================
# TOKENIZER
# ============================================================

sp = spm.SentencePieceProcessor()

if not sp.load(str(TOKENIZER_PATH)):
    raise RuntimeError(
        f"Could not load tokenizer: {TOKENIZER_PATH}"
    )

print(f"Vocabulary size: {sp.get_piece_size()}")


# ============================================================
# MODEL
# ============================================================

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

config = ModelConfig(**checkpoint["config"])

model = DecoderOnlyTransformer(config).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print(f"Checkpoint step: {checkpoint['step']}")
print(f"Parameters: {model.count_parameters():,}")


# ============================================================
# COLLECT VALIDATION SEQUENCES
# ============================================================

validation_files = sorted(
    DATA_DIR.glob("val_*.txt")
)

if not validation_files:
    raise FileNotFoundError(
        f"No validation files found in {DATA_DIR}"
    )


sequences = []

for file_path in validation_files:

    with open(
        file_path,
        "r",
        encoding="utf-8",
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            ids = sp.encode(
                line,
                out_type=int,
            )

            if len(ids) >= SEQUENCE_LENGTH:

                sequences.append(
                    ids[:SEQUENCE_LENGTH]
                )

                if len(sequences) >= NUM_SEQUENCES:
                    break

    if len(sequences) >= NUM_SEQUENCES:
        break


if len(sequences) < NUM_SEQUENCES:
    raise RuntimeError(
        f"Only found {len(sequences)} usable sequences."
    )


print(
    f"Validation sequences selected: "
    f"{len(sequences)}"
)


# ============================================================
# STORAGE
# ============================================================

entropy_values = torch.zeros(
    NUM_SEQUENCES,
    config.n_layers,
    config.n_heads,
)

distance_values = torch.zeros(
    NUM_SEQUENCES,
    config.n_layers,
    config.n_heads,
)


# ============================================================
# DISTANCE MATRIX
# ============================================================

positions = torch.arange(
    SEQUENCE_LENGTH,
    device=device,
    dtype=torch.float32,
)

distance_matrix = torch.abs(
    positions[:, None]
    - positions[None, :]
)


# ============================================================
# RUN ALL SEQUENCES
# ============================================================

for sequence_index, ids in enumerate(sequences):

    print(
        f"Processing sequence "
        f"{sequence_index + 1}/{NUM_SEQUENCES}..."
    )

    input_tensor = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device,
    )

    with torch.no_grad():

        logits, loss, attention_maps = model(
            input_tensor,
            return_attention=True,
        )

    for layer_index, attention in enumerate(
        attention_maps
    ):

        # [heads, T, T]
        attention = attention[0]

        for head_index in range(
            config.n_heads
        ):

            matrix = attention[head_index]

            # ------------------------------------------------
            # ENTROPY
            # ------------------------------------------------

            probabilities = matrix.clamp(
                min=1e-12
            )

            entropy_per_position = -(
                probabilities
                * torch.log(probabilities)
            ).sum(dim=-1)

            entropy = (
                entropy_per_position
                .mean()
                .item()
            )

            entropy_values[
                sequence_index,
                layer_index,
                head_index,
            ] = entropy

            # ------------------------------------------------
            # MEAN ATTENTION DISTANCE
            # ------------------------------------------------

            distance_per_position = (
                matrix * distance_matrix
            ).sum(dim=-1)

            distance = (
                distance_per_position
                .mean()
                .item()
            )

            distance_values[
                sequence_index,
                layer_index,
                head_index,
            ] = distance


# ============================================================
# AGGREGATE
# ============================================================

mean_entropy = entropy_values.mean(
    dim=0
)

std_entropy = entropy_values.std(
    dim=0
)

mean_distance = distance_values.mean(
    dim=0
)

std_distance = distance_values.std(
    dim=0
)


# ============================================================
# SAVE CSV
# ============================================================

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

csv_path = (
    RESULTS_DIR
    / "assamese_attention_aggregate.csv"
)

with open(
    csv_path,
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "layer",
        "head",
        "mean_entropy",
        "std_entropy",
        "mean_attention_distance",
        "std_attention_distance",
    ])

    for layer in range(config.n_layers):

        for head in range(config.n_heads):

            writer.writerow([
                layer + 1,
                head,
                f"{mean_entropy[layer, head].item():.6f}",
                f"{std_entropy[layer, head].item():.6f}",
                f"{mean_distance[layer, head].item():.6f}",
                f"{std_distance[layer, head].item():.6f}",
            ])


# ============================================================
# PRINT SUMMARY
# ============================================================

print()
print("=" * 70)
print("AGGREGATED ATTENTION STATISTICS")
print("=" * 70)

for layer in range(config.n_layers):

    print(
        f"\nLayer {layer + 1}"
    )

    for head in range(config.n_heads):

        print(
            f"  Head {head}: "
            f"Entropy = "
            f"{mean_entropy[layer, head].item():.4f} "
            f"(±{std_entropy[layer, head].item():.4f}), "
            f"Distance = "
            f"{mean_distance[layer, head].item():.4f} "
            f"(±{std_distance[layer, head].item():.4f})"
        )


# ============================================================
# ENTROPY HEATMAP
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.imshow(
    mean_entropy.numpy(),
    aspect="auto",
    interpolation="nearest",
)

plt.title(
    "assamese Mean Attention Entropy"
)

plt.xlabel(
    "Attention Head"
)

plt.ylabel(
    "Transformer Layer"
)

plt.xticks(
    range(config.n_heads)
)

plt.yticks(
    range(config.n_layers),
    range(1, config.n_layers + 1),
)

plt.colorbar(
    label="Mean Entropy"
)

plt.tight_layout()

entropy_figure = (
    FIGURES_DIR
    / "assamese_attention_entropy_heatmap.png"
)

plt.savefig(
    entropy_figure,
    dpi=200,
)

plt.close()


# ============================================================
# DISTANCE HEATMAP
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.imshow(
    mean_distance.numpy(),
    aspect="auto",
    interpolation="nearest",
)

plt.title(
    "assamese Mean Attention Distance"
)

plt.xlabel(
    "Attention Head"
)

plt.ylabel(
    "Transformer Layer"
)

plt.xticks(
    range(config.n_heads)
)

plt.yticks(
    range(config.n_layers),
    range(1, config.n_layers + 1),
)

plt.colorbar(
    label="Mean Attention Distance"
)

plt.tight_layout()

distance_figure = (
    FIGURES_DIR
    / "assamese_attention_distance_heatmap.png"
)

plt.savefig(
    distance_figure,
    dpi=200,
)

plt.close()


# ============================================================
# COMPLETE
# ============================================================

print()
print("=" * 70)
print("assamese AGGREGATED ATTENTION ANALYSIS COMPLETE")
print("=" * 70)

print(f"CSV: {csv_path}")
print(f"Entropy figure: {entropy_figure}")
print(f"Distance figure: {distance_figure}")

Writing /kaggle/working/ablation_attention_aggregate.py


In [15]:
%%writefile /kaggle/working/assamese_positional_ablation_evaluation.py
from pathlib import Path
import os
import sys
import json
import math
import time
import subprocess

import torch
import sentencepiece as spm

BASE = Path("/kaggle/working/assamese_10k")
CHECKPOINT_DIR = BASE / "checkpoints"
TOKENIZER_PATH = Path("/kaggle/working/assamese_10k.model")
DATA_DIR = Path(os.environ.get("ABLATION_DATA_DIR", "/kaggle/input/datasets/vidurshah04/assam-data"))
EVAL_DIR = BASE / "evaluation"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = EVAL_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
SCRIPTS_DIR = EVAL_DIR / "scripts"
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Locate latest checkpoint numerically, not lexicographically.
# ------------------------------------------------------------
checkpoints = sorted(
    CHECKPOINT_DIR.glob("checkpoint_*.pt"),
    key=lambda p: int(p.stem.split("_")[1]),
)
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found in {CHECKPOINT_DIR}")

CHECKPOINT_PATH = checkpoints[-1]

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(TOKENIZER_PATH)
if not DATA_DIR.exists():
    raise FileNotFoundError(DATA_DIR)

print("=" * 80)
print("ASSAMESE POSITIONAL-EMBEDDING ABLATION — PHASE 2 EVALUATION")
print("=" * 80)
print("Checkpoint :", CHECKPOINT_PATH)
print("Tokenizer  :", TOKENIZER_PATH)
print("Data       :", DATA_DIR)
print()

# ------------------------------------------------------------
# Load model once for checkpoint/validation/BPB verification.
# ------------------------------------------------------------
sys.path.insert(0, "/kaggle/working")
from config import ModelConfig
from transformer import DecoderOnlyTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)
config = ModelConfig(**checkpoint["config"])
model = DecoderOnlyTransformer(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

parameter_count = model.count_parameters()
print("Checkpoint step:", checkpoint["step"])
print("Parameters    :", parameter_count)
print("Vocabulary    :", config.vocab_size)

# Strict architecture checks for the ablation.
assert config.vocab_size == 10_000
assert not hasattr(model, "position_embedding"), "Positional embedding attribute exists."
expected_params = 25_350_912
assert parameter_count == expected_params, (
    f"Unexpected ablation parameter count: {parameter_count}; "
    f"expected {expected_params} for this architecture."
)

sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER_PATH))
assert sp.get_piece_size() == config.vocab_size
print("Tokenizer/model vocabulary match: PASS")
print("No positional embedding attribute: PASS")
print("Ablation parameter count: PASS")
print()

# ------------------------------------------------------------
# 1. Validation CE + PPL
# Matches the supplied Phase-2 evaluator protocol:
# 512-token chunks, at most 100M target tokens.
# ------------------------------------------------------------
print("=" * 80)
print("1. VALIDATION CROSS-ENTROPY + PERPLEXITY")
print("=" * 80)

SEQ_LEN = 512
MAX_VALIDATION_TOKENS = 100_000_000

total_loss = 0.0
total_tokens = 0
num_sequences = 0
token_buffer = []
started = time.time()

stop = False
with torch.no_grad():
    for file_path in sorted(DATA_DIR.glob("val_*.txt")):
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                token_buffer.extend(sp.encode(line, out_type=int))

                while len(token_buffer) >= SEQ_LEN:
                    if total_tokens >= MAX_VALIDATION_TOKENS:
                        stop = True
                        break

                    chunk = token_buffer[:SEQ_LEN]
                    del token_buffer[:SEQ_LEN]

                    x = torch.tensor([chunk[:-1]], dtype=torch.long, device=device)
                    y = torch.tensor([chunk[1:]], dtype=torch.long, device=device)

                    remaining = MAX_VALIDATION_TOKENS - total_tokens
                    if y.numel() > remaining:
                        x = x[:, :remaining]
                        y = y[:, :remaining]

                    _, loss = model(x, y)
                    n = y.numel()
                    total_loss += float(loss.item()) * n
                    total_tokens += n
                    num_sequences += 1

                    if total_tokens >= MAX_VALIDATION_TOKENS:
                        stop = True
                        break

                if stop:
                    break
        if stop:
            break

if total_tokens == 0:
    raise RuntimeError("No validation tokens were evaluated.")

validation_ce = total_loss / total_tokens
validation_ppl = math.exp(validation_ce)

validation_metrics = {
    "checkpoint": CHECKPOINT_PATH.name,
    "checkpoint_step": int(checkpoint["step"]),
    "validation_sequences": int(num_sequences),
    "validation_tokens": int(total_tokens),
    "cross_entropy": float(validation_ce),
    "perplexity": float(validation_ppl),
    "protocol_max_tokens": MAX_VALIDATION_TOKENS,
    "elapsed_minutes": (time.time() - started) / 60.0,
}
with open(EVAL_DIR / "validation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(validation_metrics, f, indent=2)

print("Validation tokens:", f"{total_tokens:,}")
print("Cross-Entropy    :", f"{validation_ce:.6f}")
print("Perplexity       :", f"{validation_ppl:.6f}")
print()

# ------------------------------------------------------------
# 2. BPB
# Matches the supplied BPB formula and uses the measured
# validation CE above.
# ------------------------------------------------------------
print("=" * 80)
print("2. BITS PER BYTE (BPB)")
print("=" * 80)

total_words = 0
total_sp_tokens = 0
total_bytes = 0

for file_path in sorted(DATA_DIR.glob("val_*.txt")):
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            total_words += len(line.split())
            total_sp_tokens += len(sp.encode(line, out_type=int))
            total_bytes += len(line.encode("utf-8"))

if total_words == 0 or total_bytes == 0:
    raise RuntimeError("Validation text is empty.")

tokens_per_word = total_sp_tokens / total_words
bpb = validation_ce * total_sp_tokens / (total_bytes * math.log(2))

bpb_metrics = {
    "checkpoint": CHECKPOINT_PATH.name,
    "validation_words": int(total_words),
    "sentencepiece_tokens": int(total_sp_tokens),
    "utf8_bytes": int(total_bytes),
    "tokens_per_word": float(tokens_per_word),
    "cross_entropy_used": float(validation_ce),
    "perplexity": float(validation_ppl),
    "bits_per_byte": float(bpb),
}
with open(EVAL_DIR / "bpb_metrics.json", "w", encoding="utf-8") as f:
    json.dump(bpb_metrics, f, indent=2)

print("Validation words     :", f"{total_words:,}")
print("SentencePiece tokens :", f"{total_sp_tokens:,}")
print("UTF-8 bytes          :", f"{total_bytes:,}")
print("Tokens / word        :", f"{tokens_per_word:.6f}")
print("BPB                  :", f"{bpb:.6f}")
print()

# Release the validation model before subprocess-based generation/attention.
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# Helper to run a generated evaluation script and save stdout.
# ------------------------------------------------------------
def run_script(script_name, log_name):
    script_path = SCRIPTS_DIR / script_name
    log_path = LOG_DIR / log_name

    env = os.environ.copy()
    env["ABLATION_CHECKPOINT"] = str(CHECKPOINT_PATH)
    env["ABLATION_EVAL_DIR"] = str(EVAL_DIR)
    env["ABLATION_DATA_DIR"] = str(DATA_DIR)

    print("=" * 80)
    print(f"RUNNING {script_name}")
    print("=" * 80)

    with open(log_path, "w", encoding="utf-8") as log:
        result = subprocess.run(
            [sys.executable, str(script_path)],
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

    if result.returncode != 0:
        print(log_path.read_text(encoding="utf-8")[-10000:])
        raise RuntimeError(f"{script_name} failed with exit code {result.returncode}")

    print(f"Completed: {script_name}")
    print(f"Log: {log_path}")

# ------------------------------------------------------------
# 3A. Standard qualitative generation
# ------------------------------------------------------------
run_script(
    "ablation_standard_generation.py",
    "standard_generation.log",
)

# ------------------------------------------------------------
# 3B. Quantitative generation metrics
# BLEU-4, chrF, ROUGE-L, repetition rate,
# Distinct-1, Distinct-2 across T=0,0.5,1,1.5.
# ------------------------------------------------------------
run_script(
    "ablation_generation_metrics.py",
    "generation_metrics.log",
)

# ------------------------------------------------------------
# 4. Aggregated attention
# 10 validation sequences, all layers x heads, entropy and
# mean attention distance with CSV + heatmaps.
# ------------------------------------------------------------
run_script(
    "ablation_attention_aggregate.py",
    "attention.log",
)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
generation_summary = EVAL_DIR / "generation_metrics" / "assamese_generation_metrics.json"

summary = {
    "experiment": "Assamese positional-embedding ablation",
    "checkpoint": CHECKPOINT_PATH.name,
    "checkpoint_step": int(checkpoint["step"]),
    "parameters": int(parameter_count),
    "validation": validation_metrics,
    "bpb": bpb_metrics,
    "generation_metrics_file": str(generation_summary),
    "standard_generation_dir": str(EVAL_DIR / "generations"),
    "attention_csv": str(EVAL_DIR / "attention" / "assamese_attention_aggregate.csv"),
    "attention_figures_dir": str(EVAL_DIR / "attention_figures"),
    "completed_at_epoch_seconds": time.time(),
}
with open(EVAL_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print()
print("=" * 80)
print("PHASE 2 ABLATION EVALUATION COMPLETE")
print("=" * 80)
print("Validation CE :", f"{validation_ce:.6f}")
print("Validation PPL:", f"{validation_ppl:.6f}")
print("BPB           :", f"{bpb:.6f}")
print("Evaluation dir:", EVAL_DIR)


Writing /kaggle/working/assamese_positional_ablation_evaluation.py


In [16]:
import py_compile
from pathlib import Path

scripts = [
    Path("/kaggle/working/ablation_standard_generation.py"),
    Path("/kaggle/working/ablation_generation_metrics.py"),
    Path("/kaggle/working/ablation_attention_aggregate.py"),
    Path("/kaggle/working/assamese_positional_ablation_evaluation.py"),
]

for script in scripts:
    py_compile.compile(str(script), doraise=True)
    print("PASS:", script.name)

print("All evaluation scripts compiled successfully.")


PASS: ablation_standard_generation.py
PASS: ablation_generation_metrics.py
PASS: ablation_attention_aggregate.py
PASS: assamese_positional_ablation_evaluation.py
All evaluation scripts compiled successfully.


In [17]:
import subprocess
import sys
from pathlib import Path

evaluation_script = Path(
    "/kaggle/working/assamese_positional_ablation_evaluation.py"
)

print("=" * 80)
print("STARTING FULL PHASE 2 ABLATION EVALUATION")
print("=" * 80)
print("Order: Validation → BPB → Standard Generation → Generation Metrics → Attention")
print()

subprocess.run(
    [sys.executable, "-u", str(evaluation_script)],
    check=True,
)

print()
print("=" * 80)
print("FULL PHASE 2 ABLATION EVALUATION RETURNED SUCCESSFULLY")
print("=" * 80)


STARTING FULL PHASE 2 ABLATION EVALUATION
Order: Validation → BPB → Standard Generation → Generation Metrics → Attention

ASSAMESE POSITIONAL-EMBEDDING ABLATION — PHASE 2 EVALUATION
Checkpoint : /kaggle/working/assamese_10k/checkpoints/checkpoint_030518.pt
Tokenizer  : /kaggle/working/assamese_10k.model
Data       : /kaggle/input/datasets/vidurshah04/assam-data

Device: cuda
GPU: Tesla T4
Checkpoint step: 30518
Parameters    : 25350912
Vocabulary    : 10000
Tokenizer/model vocabulary match: PASS
No positional embedding attribute: PASS
Ablation parameter count: PASS

1. VALIDATION CROSS-ENTROPY + PERPLEXITY
Validation tokens: 58,352,623
Cross-Entropy    : 4.970217
Perplexity       : 144.058128

2. BITS PER BYTE (BPB)
Validation words     : 32,466,286
SentencePiece tokens : 58,466,926
UTF-8 bytes          : 530,517,951
Tokens / word        : 1.800850
BPB                  : 0.790242

RUNNING ablation_standard_generation.py
/usr/bin/python3: can't open file '/kaggle/working/assamese_10k/ev

Traceback (most recent call last):
  File "/kaggle/working/assamese_positional_ablation_evaluation.py", line 264, in <module>
    run_script(
  File "/kaggle/working/assamese_positional_ablation_evaluation.py", line 256, in run_script
    raise RuntimeError(f"{script_name} failed with exit code {result.returncode}")
RuntimeError: ablation_standard_generation.py failed with exit code 2


CalledProcessError: Command '['/usr/bin/python3', '-u', '/kaggle/working/assamese_positional_ablation_evaluation.py']' returned non-zero exit status 1.

In [ ]:
%%writefile /kaggle/working/package_assamese_ablation.py
from pathlib import Path
import shutil
import json
import os
import zipfile

BASE = Path("/kaggle/working/assamese_10k")
FINAL = Path("/kaggle/working/ASSAMESE_POSITIONAL_ABLATION_FINAL")

if not (BASE / "evaluation" / "summary.json").exists():
    raise FileNotFoundError("Evaluation has not completed successfully.")

if FINAL.exists():
    shutil.rmtree(FINAL)
FINAL.mkdir(parents=True)

# ------------------------------------------------------------
# Core experiment files
# ------------------------------------------------------------
for filename in [
    "assamese_10k.model",
    "config.py",
    "transformer.py",
    "dataset.py",
    "train_assamese_10k.py",
]:
    src = Path("/kaggle/working") / filename
    if not src.exists():
        raise FileNotFoundError(src)
    shutil.copy2(src, FINAL / filename)

# ------------------------------------------------------------
# Training logs + completion metadata
# ------------------------------------------------------------
for rel in [
    Path("logs") / "training_log.csv",
    Path("training_complete.json"),
]:
    src = BASE / rel
    if src.exists():
        dst = FINAL / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

# ------------------------------------------------------------
# Evaluation tree — copy everything.
# ------------------------------------------------------------
shutil.copytree(
    BASE / "evaluation",
    FINAL / "evaluation",
)

# ------------------------------------------------------------
# Checkpoints:
# Keep the latest 5 numeric checkpoints in the final package.
# The runtime still has the full checkpoint history in BASE.
# ------------------------------------------------------------
checkpoints = sorted(
    (BASE / "checkpoints").glob("checkpoint_*.pt"),
    key=lambda p: int(p.stem.split("_")[1]),
)
if not checkpoints:
    raise FileNotFoundError("No training checkpoints found.")

ckpt_dir = FINAL / "checkpoints"
ckpt_dir.mkdir(parents=True)

for ckpt in checkpoints[-5:]:
    shutil.copy2(ckpt, ckpt_dir / ckpt.name)

selected = checkpoints[-1]

# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------
manifest = {
    "experiment": "Assamese positional-embedding ablation",
    "selected_checkpoint": selected.name,
    "selected_checkpoint_step": int(selected.stem.split("_")[1]),
    "latest_checkpoints_in_package": [p.name for p in checkpoints[-5:]],
    "all_runtime_checkpoints": [p.name for p in checkpoints],
    "training_target_tokens": 500_000_000,
    "sequence_length": 512,
    "batch_size": 1,
    "gradient_accumulation": 32,
    "tokens_per_optimizer_step": 16_384,
    "optimizer_steps": 30_518,
    "learning_rate": 3e-4,
    "min_learning_rate": 3e-5,
    "warmup_steps": 2_000,
    "weight_decay": 0.1,
    "adam_betas": [0.9, 0.95],
    "grad_clip": 1.0,
    "dropout": 0.1,
    "mixed_precision": "float16 AMP",
    "positional_embeddings": False,
}
with open(FINAL / "experiment_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# README
# ------------------------------------------------------------
readme = f"""ASSAMESE POSITIONAL-EMBEDDING ABLATION
==========================================

Selected checkpoint:
{selected.name}

Checkpoint step:
{selected.stem.split('_')[1]}

This package contains:
- tokenizer
- ablation transformer/config/dataset
- training script
- latest 5 checkpoints
- training log
- Phase 2 validation CE/PPL
- BPB
- standard generation samples
- quantitative generation metrics
- attention aggregate CSV
- attention entropy/distance heatmaps
- evaluation logs
- experiment manifest

The full runtime checkpoint history remains under:
/kaggle/working/assamese_10k/checkpoints

The final checkpoint used for evaluation is:
{selected}

Important:
The positional-embedding ablation model intentionally contains token
embeddings only; positional embeddings are not added in forward().
"""
with open(FINAL / "README.txt", "w", encoding="utf-8") as f:
    f.write(readme)

# ------------------------------------------------------------
# Zip the final package.
# ------------------------------------------------------------
archive = shutil.make_archive(
    str(Path("/kaggle/working/ASSAMESE_POSITIONAL_ABLATION_FINAL")),
    "zip",
    root_dir=FINAL,
)

print("=" * 80)
print("FINAL OUTPUT PACKAGE CREATED")
print("=" * 80)
print("Directory:", FINAL)
print("ZIP      :", archive)
print()

files = [p for p in FINAL.rglob("*") if p.is_file()]
print("Files in final package:", len(files))
total_size = sum(p.stat().st_size for p in files)
print("Total package size:", f"{total_size / (1024**3):.2f} GB")
print()

for p in sorted(files):
    print(f"{p.relative_to(FINAL)}\t{p.stat().st_size / (1024**2):.2f} MB")


In [ ]:
import subprocess
import sys
from pathlib import Path

packager = Path("/kaggle/working/package_assamese_ablation.py")

subprocess.run(
    [sys.executable, "-u", str(packager)],
    check=True,
)


In [ ]:

# Optional: upload the final checkpoint/package to a Kaggle Dataset.
# Disabled by default because Kaggle API uploads require network access
# and write permission to the target dataset.

ENABLE_KAGGLE_DATASET_UPLOAD = False
KAGGLE_DATASET_ID = "sonushah26/assamese-pos-ablation"

if ENABLE_KAGGLE_DATASET_UPLOAD:
    import subprocess
    from pathlib import Path
    import json

    upload_dir = Path("/kaggle/working/assamese_ablation_ckpt_upload")
    upload_dir.mkdir(parents=True, exist_ok=True)

    final_checkpoint = sorted(
        Path("/kaggle/working/assamese_10k/checkpoints").glob("checkpoint_*.pt"),
        key=lambda p: int(p.stem.split("_")[1])
    )[-1]

    shutil.copy2(
        final_checkpoint,
        upload_dir / "assamese_positional_ablation_final.pt",
    )

    metadata = {
        "title": "Assamese LMA Positional Ablation Checkpoint",
        "id": KAGGLE_DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(upload_dir / "dataset-metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    result = subprocess.run(
        [
            "kaggle", "datasets", "version",
            "-p", str(upload_dir),
            "-m", "Assamese positional ablation checkpoint",
        ],
        text=True,
    )

    if result.returncode == 0:
        print("Kaggle Dataset upload completed.")
    else:
        print("WARNING: Kaggle Dataset upload failed; local final outputs are intact.")
else:
    print("Kaggle Dataset upload: SKIPPED (disabled by default).")


## Expected final files

After successful completion, the main package is:

`/kaggle/working/ASSAMESE_POSITIONAL_ABLATION_FINAL/`

and the ZIP is:

`/kaggle/working/ASSAMESE_POSITIONAL_ABLATION_FINAL.zip`

The evaluation directory contains:
- `validation_metrics.json`
- `bpb_metrics.json`
- standard generation text files
- quantitative generation metrics JSON/CSV
- attention aggregate CSV
- attention entropy/distance heatmaps
- evaluation logs

The package includes the latest five checkpoints plus the complete checkpoint-history list in `experiment_manifest.json`.

### Remote-run instruction

Use **Save Version → Save & Run All** in Kaggle with a **Tesla T4 GPU**. Once the versioned run is actually running, you can close the browser/laptop without cancelling the Kaggle session.
